# Convert to CSV

In [ ]:
import re
import csv

def parse_output(input_file):
    output_file = f"{input_file.split('.txt')[0]}.csv"

    # --- Hardware maps ---
    pcore_map = {
        "xs-4114": 20,
        "dxs-4114": 40,
        "i7-7700": 8,
        "i7-9700": 8,
        "i7-13700": 16,
        "w5-3423": 24
    }

    # Reverse mapping: node -> hardware
    node_to_hw = {
        '008': 'xs-4114',
        '012': 'i7-7700',
        '015': 'i7-7700',
        '016': 'i7-7700',
        '018': 'dxs-4114',
        '019': 'dxs-4114',
        '020': 'i7-9700',
        '027': 'w5-3423',
        '029': 'w5-3423',
        '030': 'w5-3423',
        '031': 'w5-3423',
        '032': 'w5-3423',
        '033': 'i7-13700',
        '034': 'i7-13700',
        '035': 'i7-13700',
        '036': 'i7-13700',
        '037': 'i7-13700',
        '038': 'i7-13700',
        '039': 'i7-13700',
        '040': 'i7-13700'
    }

    # --- Regex ---
    config_pattern = re.compile(r"N=(\d+), Q=(\d+), A=(\d+), K=(\d+), ARGS=\[(.*?)\]")

    # Updated to capture "033&008" directly from the custom log line
    node_pattern = re.compile(r"Nodes:\s+\[(.*?)\]")

    perf_pattern = re.compile(r"\d+\.\s+(.+?)\s+:\s+([\d.]+)\s+s")
    total_pattern = re.compile(r"Total Time\s+:\s+([\d.]+)\s+s")

    # --- Read file ---
    with open(input_file, "r") as f:
        lines = f.readlines()

    rows_out = []

    # Track dynamic performance columns
    dynamic_perf_keys = []

    # Stateful variables to hold properties until a successful run writes them
    current_config = None
    current_args = None
    current_nodes_str = None
    total_p_threads = 0

    i = 0
    while i < len(lines):
        line = lines[i]

        # --- Extract Config ---
        m = config_pattern.search(line)
        if m:
            N, Q, A, K, args_str = m.groups()
            current_config = (int(N), int(Q), int(A), int(K))

            # Split the string of arguments by whitespace
            args_parts = args_str.strip().split()

            # Apply assignment rules and defaults based on length
            d_split = int(args_parts[0]) if len(args_parts) > 0 else 1
            data_row = int(args_parts[1]) if len(args_parts) > 1 else 0
            mode = int(args_parts[2]) if len(args_parts) > 2 else 0

            current_args = (d_split, data_row, mode)

        # --- Extract Nodes (Updated Logic) ---
        node_match = node_pattern.search(line)
        if node_match:
            raw_nodes = node_match.group(1) # e.g., "033&008"
            nodes = raw_nodes.split("&")
            current_nodes_str = f"[{raw_nodes}]"

            # Calculate total threads strictly from the nodes we just found
            total_p_threads = 0
            for node in nodes:
                hw = node_to_hw.get(node)
                if hw:
                    total_p_threads += pcore_map.get(hw, 0)

        # --- Parse performance block ---
        if "--- MPI Rank 0 Performance Profile ---" in line:
            perf_dict = {}
            total_time = 0.0

            i += 1
            while i < len(lines):
                l = lines[i]

                # total time
                total_match = total_pattern.search(l)
                if total_match:
                    total_time = float(total_match.group(1))
                    break

                # perf entries
                perf_match = perf_pattern.search(l)
                if perf_match:
                    name = perf_match.group(1).strip().lower().replace(" ", "_")
                    value = float(perf_match.group(2))
                    perf_dict[name] = value

                    # Add new metric to our tracker if we haven't seen it yet
                    if name not in dynamic_perf_keys:
                        dynamic_perf_keys.append(name)

                i += 1

            # --- Build row ---
            d_split, data_row, mode = current_args

            row_dict = {
                "nodes": current_nodes_str,
                "p_threads": total_p_threads,
                "num_datapoints": current_config[0],
                "num_queries": current_config[1],
                "num_attrs": current_config[2],
                "avg_neighbours": current_config[3],
                "data_split": d_split,
                "data_row": data_row,
                "mode": mode,
                "total_time": total_time
            }

            # Merge dynamically extracted timings into the row dictionary
            row_dict.update(perf_dict)
            rows_out.append(row_dict)

        i += 1

    # --- Write CSV dynamically ---
    base_header = [
        "nodes", "p_threads", "num_datapoints",
        "num_queries", "num_attrs", "avg_neighbours",
        "data_split", "data_row", "mode"
    ]

    # Final header is base + dynamic performance metrics + total time
    final_header = base_header + dynamic_perf_keys + ["total_time"]

    with open(output_file, "w", newline="") as f:
        # csv.DictWriter allows us to pass a dictionary and automatically maps keys to columns
        writer = csv.DictWriter(f, fieldnames=final_header)
        writer.writeheader()
        writer.writerows(rows_out)

    print(f"CSV written to {output_file}")

# Example execution:
# parse_output("my_log_file.txt")

# Process for other funcs here

In [1]:
import re
import pandas as pd

hw_to_nodes = {
    "xs-4114": ["008"],
    "i7-7700": ["012", "015", "016"],
    "dxs-4114": ["018", "019"],
    "i7-9700": ["020"],
    "w5-3423": ["027", "029", "030", "031", "032"],
    "i7-13700": ["033", "034", "035", "036", "037", "038", "039", "040"]
}

node_to_hw = {
    '008': 'xs-4114',
    '012': 'i7-7700',
    '015': 'i7-7700',
    '016': 'i7-7700',
    '018': 'dxs-4114',
    '019': 'dxs-4114',
    '020': 'i7-9700',
    '027': 'w5-3423',
    '029': 'w5-3423',
    '030': 'w5-3423',
    '031': 'w5-3423',
    '032': 'w5-3423',
    '033': 'i7-13700',
    '034': 'i7-13700',
    '035': 'i7-13700',
    '036': 'i7-13700',
    '037': 'i7-13700',
    '038': 'i7-13700',
    '039': 'i7-13700',
    '040': 'i7-13700'
}

# hw_rel_comp_bf = {"xs-4114": 1.42, "i7-7700": 0.91, "i7-9700": 0.67, "i7-13700": 0.77, "w5-3423": 0.89, "dxs-4114": 1.33}
# hw_rel_comp_kd = {"xs-4114": 1.49, "i7-7700": 0.94, "i7-9700": 0.54, "i7-13700": 0.67, "w5-3423": 0.84, "dxs-4114": 1.51}

hw_cores = {"xs-4114": 20, "i7-7700": 8, "i7-9700": 8, "i7-13700": 16, "w5-3423": 24, "dxs-4114": 40}

node_to_cores = {
    "008": 20, "012": 8, "015": 8, "016": 8, "018": 40, "019": 40,
    "020": 8, "027": 24, "029": 24, "030": 24, "031": 24, "032": 24,
    "033": 16, "034": 16, "035": 16, "036": 16, "037": 16, "038": 16,
    "039": 16, "040": 16
}

# # 1. Load the CSV
# # index_col=0 treats the first column as the row IDs
# df = pd.read_csv('latencies.csv', index_col=0)

# # Optional: Ensure all IDs (index and columns) are strings to maintain formatting (like '008')
# df.index = df.index.map(lambda x: str(x).zfill(3))
# df.columns = df.columns.map(lambda x: str(x).zfill(3))

# # 2. Process into a dictionary: dict[('id1', 'id2'), value]
# # .stack() turns the 2D table into a Series with a MultiIndex (row_id, col_id)
# pairings = df.stack().to_dict()

# 3. Process into a 2D array (List of Lists)
# latencies = df.values.tolist()

# # --- Verification ---
# # print(f"Dictionary sample (first item): {list(pairings.items())[0]}")

# # def generate_and_print_node_dicts(node_to_hw, hw_rel_comp_bf, hw_rel_comp_kd):
# #     # 1. Generate the mapped dictionaries
# #     node_to_comp_bf = {node: hw_rel_comp_bf[hw] for node, hw in node_to_hw.items()}
# #     node_to_comp_kd = {node: hw_rel_comp_kd[hw] for node, hw in node_to_hw.items()}

# #     # 2. Formatter for copy-pasting (20 items per row)
# #     def print_chunked(dict_name, my_dict, items_per_row=20):
# #         items = [f"{repr(k)}: {repr(v)}" for k, v in my_dict.items()]
# #         chunks = [items[i:i + items_per_row] for i in range(0, len(items), items_per_row)]
# #         lines = [", ".join(chunk) for chunk in chunks]
# #         dict_str = f"{dict_name} = {{\n    " + ",\n    ".join(lines) + "\n}"
# #         print(dict_str)
# #         print("\n" + "="*50 + "\n") # Separator

# #     # 3. Print them out
# #     print_chunked("node_to_comp_bf", node_to_comp_bf)
# #     print_chunked("node_to_comp_kd", node_to_comp_kd)

# # --- Example Usage ---
# node_to_comp_bf = {
#     '008': 1.42, '015': 0.91, '016': 0.91, '012': 0.91, '018': 1.33, '019': 1.33, '020': 0.67, '029': 0.89, '030': 0.89, '031': 0.89, '032': 0.89, '027': 0.89, '034': 0.77, '035': 0.77, '033': 0.77, '036': 0.77, '037': 0.77, '038': 0.77, '039': 0.77, '040': 0.77
# }

# node_to_comp_kd = {
#     '008': 1.49, '015': 0.94, '016': 0.94, '012': 0.94, '018': 1.51, '019': 1.51, '020': 0.54, '029': 0.84, '030': 0.84, '031': 0.84, '032': 0.84, '027': 0.84, '034': 0.67, '035': 0.67, '033': 0.67, '036': 0.67, '037': 0.67, '038': 0.67, '039': 0.67, '040': 0.67
# }

In [29]:
import torch

node_to_id = {
    '008': 0,
    '012': 1,
    '015': 2,
    '016': 3,
    '018': 4,
    '019': 5,
    '020': 6,
    '027': 7,
    '029': 8,
    '030': 9,
    '031': 10,
    '032': 11,
    '033': 12,
    '034': 13,
    '035': 14,
    '036': 15,
    '037': 16,
    '038': 17,
    '039': 18,
    '040': 19
    }

node_comp_bf = torch.tensor([1.6207, 1.1694, 1.1694, 1.1694, 1.5697, 1.5697, 0.6403, 1.5281, 1.5281,
        1.5281, 1.5281, 1.5281, 0.9086, 0.9086, 0.9086, 0.9086, 0.9086, 0.9086,
        0.9086, 0.9086])
node_comp_kd = torch.tensor([0.5386, 0.9750, 0.9750, 0.9750, 0.8193, 0.8193, 0.3643, 1.9103, 1.9103,
        1.9103, 1.9103, 1.9103, 0.5825, 0.5825, 0.5825, 0.5825, 0.5825, 0.5825,
        0.5825, 0.5825])

node_pair_bandwidth = torch.tensor([[6.2085e-02, 6.4740e-01, 1.6800e-01, 1.4071e+00, 5.5790e-01, 2.2007e+00,
         9.4325e-01, 2.9655e-01, 3.4440e-01, 8.1160e-01, 5.5790e-01, 2.9280e-01,
         3.1020e-01, 4.2230e-01, 1.4071e+00, 1.1859e+00, 8.6830e-01, 7.9350e-01,
         1.2211e+00, 2.3620e-01],
        [6.4740e-01, 2.1090e-02, 1.5775e-01, 1.6271e-01, 2.4020e-01, 2.0722e+00,
         2.9655e-01, 4.2340e-01, 4.7014e-01, 5.2423e-01, 5.6808e-01, 3.5005e-01,
         2.3309e-02, 5.3205e-01, 4.6413e+00, 1.3192e+00, 2.6005e-02, 2.3323e-02,
         3.1173e-02, 4.1091e-02],
        [1.6800e-01, 1.5775e-01, 2.0463e-02, 1.9136e-01, 1.3130e-01, 1.8388e+00,
         4.3390e-01, 4.3952e-01, 4.7742e-01, 4.5469e-01, 1.2278e+00, 4.9710e-01,
         3.6111e-01, 8.7762e-01, 3.5439e+00, 2.2948e+00, 3.9937e-02, 3.6397e-02,
         3.7840e-02, 4.6340e-02],
        [1.4071e+00, 1.6271e-01, 1.9136e-01, 1.8857e-02, 2.9390e-01, 2.1405e+00,
         6.9390e-01, 1.0675e+00, 2.9334e-01, 4.5526e-01, 4.6536e-01, 4.4031e-01,
         1.5614e-01, 1.8426e-01, 5.1118e+00, 1.3734e+00, 4.1863e-02, 1.0277e-01,
         8.7763e-02, 1.2987e-01],
        [5.5790e-01, 2.4020e-01, 1.3130e-01, 2.9390e-01, 5.5165e-02, 1.2662e+00,
         2.9660e-01, 3.9125e-01, 6.0520e-01, 4.6225e-01, 6.8490e-01, 9.7030e-01,
         2.4160e-01, 2.1860e-01, 9.2520e-01, 9.7030e-01, 1.4300e-01, 4.7890e-01,
         6.2380e-01, 7.2250e-01],
        [2.2007e+00, 2.0722e+00, 1.8388e+00, 2.1405e+00, 1.2662e+00, 8.6828e-02,
         1.5791e+00, 1.6579e+00, 1.5186e+00, 1.0206e+00, 2.1294e+00, 1.1551e+00,
         1.4265e+00, 2.6118e+00, 2.7570e-01, 3.8200e-01, 2.0041e+00, 1.4363e+00,
         1.4653e+00, 1.5194e+00],
        [9.4325e-01, 2.9655e-01, 4.3390e-01, 6.9390e-01, 2.9660e-01, 1.5791e+00,
         1.8074e-02, 3.8445e-01, 3.0475e-01, 1.8620e-01, 1.8115e-01, 1.8760e-01,
         3.3480e-01, 2.4040e-01, 9.4000e-01, 2.8235e+00, 3.8340e-01, 4.4250e-01,
         3.7090e-01, 2.6220e-01],
        [2.9655e-01, 4.2340e-01, 4.3952e-01, 1.0675e+00, 3.9125e-01, 1.6579e+00,
         3.8445e-01, 8.5909e-02, 7.2438e-02, 1.0025e-01, 1.1212e-01, 1.0215e-01,
         2.3403e-01, 1.0000e-12, 3.0749e+00, 3.3210e+00, 1.7837e-01, 1.6188e-01,
         1.7851e-01, 1.8652e-01],
        [3.4440e-01, 4.7014e-01, 4.7742e-01, 2.9334e-01, 6.0520e-01, 1.5186e+00,
         3.0475e-01, 7.2438e-02, 5.1807e-02, 8.1689e-02, 8.5319e-02, 6.1983e-02,
         1.2290e-01, 5.4523e-02, 2.0933e+00, 3.1717e+00, 1.0032e-01, 1.8826e-01,
         2.1090e-01, 2.2436e-01],
        [8.1160e-01, 5.2423e-01, 4.5469e-01, 4.5526e-01, 4.6225e-01, 1.0206e+00,
         1.8620e-01, 1.0025e-01, 8.1689e-02, 5.6507e-02, 1.3573e-01, 7.3461e-02,
         1.8714e-01, 1.5459e-01, 3.0422e+00, 3.1247e+00, 2.1228e-01, 2.0341e-01,
         1.9642e-01, 3.9087e-01],
        [5.5790e-01, 5.6808e-01, 1.2278e+00, 4.6536e-01, 6.8490e-01, 2.1294e+00,
         1.8115e-01, 1.1212e-01, 8.5319e-02, 1.3573e-01, 6.8329e-02, 1.1598e-01,
         1.0442e+00, 1.9662e-01, 3.2564e+00, 3.1325e+00, 1.8591e-01, 1.2041e-01,
         1.5874e-01, 2.1269e-01],
        [2.9280e-01, 3.5005e-01, 4.9710e-01, 4.4031e-01, 9.7030e-01, 1.1551e+00,
         1.8760e-01, 1.0215e-01, 6.1983e-02, 7.3461e-02, 1.1598e-01, 4.0150e-02,
         1.4926e+00, 1.3243e-01, 3.2991e+00, 3.3183e+00, 2.0119e-01, 1.6391e-01,
         1.3222e-01, 2.2622e-01],
        [3.1020e-01, 2.3309e-02, 3.6111e-01, 1.5614e-01, 2.4160e-01, 1.4265e+00,
         3.3480e-01, 2.3403e-01, 1.2290e-01, 1.8714e-01, 1.0442e+00, 1.4926e+00,
         3.1160e-02, 3.7539e-01, 4.9017e+00, 5.0682e+00, 4.0687e-01, 4.4401e-01,
         4.0985e-01, 3.7693e-01],
        [4.2230e-01, 5.3205e-01, 8.7762e-01, 1.8426e-01, 2.1860e-01, 2.6118e+00,
         2.4040e-01, 1.0000e-12, 5.4523e-02, 1.5459e-01, 1.9662e-01, 1.3243e-01,
         3.7539e-01, 2.4485e-02, 4.9918e+00, 4.9769e+00, 4.2174e-01, 3.6032e-01,
         3.4805e-01, 3.8274e-01],
        [1.4071e+00, 4.6413e+00, 3.5439e+00, 5.1118e+00, 9.2520e-01, 2.7570e-01,
         9.4000e-01, 3.0749e+00, 2.0933e+00, 3.0422e+00, 3.2564e+00, 3.2991e+00,
         4.9017e+00, 4.9918e+00, 2.9714e-02, 3.6014e-01, 5.2892e+00, 5.2301e+00,
         5.4428e+00, 5.2144e+00],
        [1.1859e+00, 1.3192e+00, 2.2948e+00, 1.3734e+00, 9.7030e-01, 3.8200e-01,
         2.8235e+00, 3.3210e+00, 3.1717e+00, 3.1247e+00, 3.1325e+00, 3.3183e+00,
         5.0682e+00, 4.9769e+00, 3.6014e-01, 2.7596e-02, 5.0971e+00, 4.9839e+00,
         5.0721e+00, 5.0181e+00],
        [8.6830e-01, 2.6005e-02, 3.9937e-02, 4.1863e-02, 1.4300e-01, 2.0041e+00,
         3.8340e-01, 1.7837e-01, 1.0032e-01, 2.1228e-01, 1.8591e-01, 2.0119e-01,
         4.0687e-01, 4.2174e-01, 5.2892e+00, 5.0971e+00, 3.1108e-02, 3.9347e-01,
         4.0216e-01, 3.9893e-01],
        [7.9350e-01, 2.3323e-02, 3.6397e-02, 1.0277e-01, 4.7890e-01, 1.4363e+00,
         4.4250e-01, 1.6188e-01, 1.8826e-01, 2.0341e-01, 1.2041e-01, 1.6391e-01,
         4.4401e-01, 3.6032e-01, 5.2301e+00, 4.9839e+00, 3.9347e-01, 3.1965e-02,
         3.2472e-01, 3.8568e-01],
        [1.2211e+00, 3.1173e-02, 3.7840e-02, 8.7763e-02, 6.2380e-01, 1.4653e+00,
         3.7090e-01, 1.7851e-01, 2.1090e-01, 1.9642e-01, 1.5874e-01, 1.3222e-01,
         4.0985e-01, 3.4805e-01, 5.4428e+00, 5.0721e+00, 4.0216e-01, 3.2472e-01,
         2.8069e-02, 3.8246e-01],
        [2.3620e-01, 4.1091e-02, 4.6340e-02, 1.2987e-01, 7.2250e-01, 1.5194e+00,
         2.6220e-01, 1.8652e-01, 2.2436e-01, 3.9087e-01, 2.1269e-01, 2.2622e-01,
         3.7693e-01, 3.8274e-01, 5.2144e+00, 5.0181e+00, 3.9893e-01, 3.8568e-01,
         3.8246e-01, 3.0119e-02]])

# node_pair_bandwidth = torch.ones_like(node_pair_bandwidth) - torch.eye(20)

# Train

In [30]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Make sure these are defined globally in your notebook!
# node_to_comp_bf, node_to_comp_kd, node_pair_latencies, node_to_cores, node_to_id
MAX_NODES = len(node_to_id) # Adjust this to match the size of your node_to_id dictionary
MAX_BCAST_LINKS = 48

class HPCAnalyticalModel(nn.Module):
    def __init__(self):
        super(HPCAnalyticalModel, self).__init__()

        # --- 1. Topology Parameters ---
        self.node_comp_bf = nn.Parameter(node_comp_bf, requires_grad=False)
        self.node_comp_kd = nn.Parameter(node_comp_kd, requires_grad=False)

        # Sole Network Matrix (Bandwidth)
        self.node_pair_bandwidth = nn.Parameter(node_pair_bandwidth, requires_grad=False)

        # --- 2. Network Coefficients ---
        self.w_t1 = nn.Parameter(torch.tensor([0.2044]), requires_grad=True)

        # --- 3. Computation Coefficients ---
        self.w_c_bf = nn.Parameter(torch.tensor([1.1305]), requires_grad=False)
        self.w_c_kd = nn.Parameter(torch.tensor([0.1762]), requires_grad=False)

        # --- 4. Tree Build Coefficients ---
        self.w_b_kd = nn.Parameter(torch.tensor([0.7094]), requires_grad=False)

        # --- 5. Reduction/Sendback ---
        self.w_r1 = nn.Parameter(torch.tensor([0.1389]), requires_grad=True)
        self.w_r2 = nn.Parameter(torch.tensor([0.5702]), requires_grad=True)
        self.w_s1 = nn.Parameter(torch.tensor([1.374]), requires_grad=True)

    def _matmul_dynamic_max(self, matrix, flat_bandwidths, tau=10.0):
        """
        1. MATMUL the N x 400 matrix with the 400 bandwidth vector
        2. Dynamic Max: Uses Softmax for training gradients, Hard Max for deployment accuracy.
        """
        # Step 1: MATMUL
        # [Batch, N_Vecs, 400] @ [400] -> [Batch, N_Vecs]
        link_times = torch.matmul(matrix, flat_bandwidths)

        # Step 2: Mask padded rows (Find rows where the vector sum > 0)
        active_rows = (matrix.sum(dim=2) > 0)

        # Replace padded rows with a massive negative number so they equal 0 in Softmax/Max
        safe_link_times = torch.where(active_rows, link_times, torch.tensor(-1e9, device=flat_bandwidths.device))

        # Step 3: DYNAMIC ROUTING
        if self.training:
            # Training Mode: Differentiable LogSumExp to distribute gradients
            res = torch.logsumexp(safe_link_times * tau, dim=1) / tau
        else:
            # Deployment Mode: Actual hard maximum
            res = torch.max(safe_link_times, dim=1)[0]

        # Step 4: ABSOLUTE FLOOR
        # Physically guarantees network time cannot be negative.
        # Instantly neutralizes the -1e9 edge case if all rows were masked.
        return torch.clamp(res, min=0.0)

    def forward(self, x, topo_tensors):
        N_loc = x[:, 0]
        Q_loc = x[:, 1]
        A     = x[:, 2]
        K     = x[:, 3]
        d_split = x[:, 4]
        q_split = x[:, 5]
        mode  = x[:, 6]

        m0 = (mode == 0).float()
        m1 = (mode == 1).float()

        # Unpack the 2D Broadcast Vectors
        c_dp, bv_dp, pen_dp = topo_tensors['dp']
        c_q, bv_q, pen_q    = topo_tensors['q']
        c_sb                = topo_tensors['sb']
        c_red               = topo_tensors['red']
        node_mask           = topo_tensors['node_mask']

        # --- SYMMETRY FORCE ---
        sym_bandwidths = (self.node_pair_bandwidth + self.node_pair_bandwidth.t()) / 2.0
        flat_bandwidths = sym_bandwidths.reshape(-1)

        # ==========================================
        # --- VECTORIZED NETWORK COMMUNICATION ---
        # ==========================================

        # 1. Database (N) Distribution
        t_dp_scatter = N_loc * torch.matmul(c_dp, flat_bandwidths)
        # Uses the dynamic switcher automatically!
        t_dp_bcast   = N_loc * pen_dp * self._matmul_dynamic_max(bv_dp, flat_bandwidths)
        # FLAT BROADCAST (Removed tree_steps_dp)
        U_t_dp = t_dp_scatter + t_dp_bcast

        # 2. Query (Q) Distribution
        t_q_scatter = Q_loc * torch.matmul(c_q, flat_bandwidths)
        # Uses the dynamic switcher automatically!
        t_q_bcast   = Q_loc * pen_q * self._matmul_dynamic_max(bv_q, flat_bandwidths)
        # FLAT BROADCAST (Removed tree_steps_q)
        U_t_q = t_q_scatter + t_q_bcast

        # 3. Sendback (Gather)
        U_t_sb = Q_loc * torch.matmul(c_sb, flat_bandwidths)
        U_t_red = Q_loc * self._matmul_dynamic_max(c_red, flat_bandwidths)

        # ==========================================
        # --- VECTORIZED COMPUTATION ---
        # ==========================================
        U_t_bf = torch.max(self.node_comp_bf.unsqueeze(0) * node_mask, dim=1)[0]
        U_t_kd = torch.max(self.node_comp_kd.unsqueeze(0) * node_mask, dim=1)[0]

        # ==========================================
        # --- FINAL SCALING ---
        # ==========================================
        U_t_dp_millions = U_t_dp / 1e6
        U_t_q_millions  = U_t_q / 1e6

        t_sd = self.w_t1 * (A + 0.5) * U_t_dp_millions
        t_sq = self.w_t1 * (A + 0.5) * U_t_q_millions

        n_log_n_millions = (N_loc * torch.log2(N_loc)) / 1e8
        t_bt = m1 * (U_t_kd * self.w_b_kd * A * n_log_n_millions)

        comp_bf = self.w_c_bf * U_t_bf * A * Q_loc * N_loc / 1e9
        comp_kd = self.w_c_kd * U_t_kd * A * Q_loc * N_loc / 1e8
        t_comp = m0 * comp_bf + m1 * comp_kd

        red_comm = self.w_r1 * U_t_red * pen_q * K / 1e7
        red_comp = self.w_r2 * Q_loc * torch.log2(d_split) * K / 1e7

        t_red = red_comm + red_comp
        t_sb = self.w_s1 * K * U_t_sb / 1e5

        print(U_t_dp, U_t_q, U_t_red, U_t_sb)

        return torch.stack([t_sd, t_sq, t_bt, t_comp, t_red, t_sb], dim=1)


# --- PREPROCESSING LOGIC ---
def precompute_topology(row, get_pen=True):
    nodes_str = row['nodes']
    d_split = row['data_split']
    q_split = row['query_split']
    flip = bool(row['flip'])

    if get_pen:
        local_d = row['local_datapoints']
        local_q = row['local_queries']
        t_d = row['send_data']
        t_q = row['send_query']

    node_list = nodes_str.strip('[]').split('&')
    master_node = node_list[0]

    flat_ids = []
    max_remote_cores = 0
    for n in node_list:
        cores = node_to_cores[n]
        nid = node_to_id[n]
        flat_ids.extend([nid] * cores)
        if n != master_node:
            max_remote_cores = max(max_remote_cores, cores)

    total_pcores = len(flat_ids)

    def simulate_grid(is_col, apply_pen, is_sb=False):
        counts = np.zeros(MAX_NODES * MAX_NODES, dtype=np.float32)
        # CREATE N x 400 MATRIX INSTEAD OF 1D MASK
        vecs = np.zeros((MAX_BCAST_LINKS, MAX_NODES * MAX_NODES), dtype=np.float32)

        pen = 1.0
        if max_remote_cores > 0: pen = float(max_remote_cores)
        if (not flip and d_split == 1) or (flip and d_split == total_pcores): pen = 1.0
        if not apply_pen: pen = 1.0

        if total_pcores == 0 or d_split == 0 or total_pcores % d_split != 0:
            return counts, vecs, pen

        rows, cols = (int(d_split), int(total_pcores // d_split)) if flip else (int(total_pcores // d_split), int(d_split))
        matrix = np.array(flat_ids).reshape(rows, cols)
        mat = matrix.T if is_col else matrix
        r_count, c_count = mat.shape

        if r_count == 0 or c_count == 0:
            return counts, vecs, pen

        # Scatter Accumulation
        for c in range(1, c_count):
            id1, id2 = mat[0, 0], mat[0, c]
            counts[id1 * MAX_NODES + id2] += 1

        # N-VECTOR BROADCAST ASSIGNMENT
        idx = 0

        # tree reduce
        if is_sb:
            for c in range(c_count):
                step_size = 1

                # FIX 1: < r_count so the final step to root is actually recorded
                while step_size < r_count:
                    for r in range(0, r_count - step_size, 2 * step_size):
                        if idx >= MAX_BCAST_LINKS:
                            break # Safety bound

                        # FIX 2: Sender is r+step_size, Receiver is r (flows to root)
                        id1, id2 = mat[r + step_size, c], mat[r, c]

                        # ASSIGN TO THE SAME VECTOR: Entire tree evaluates in parallel
                        vecs[idx, id1 * MAX_NODES + id2] += 1.0

                    step_size *= 2

                # INCREMENT HERE: One vector per complete reduction column
                idx += 1

        if not is_sb:
            for c in range(c_count):
                for r in range(1, r_count):
                    if idx >= MAX_BCAST_LINKS:
                        break # Safety bound
                    id1, id2 = mat[0, c], mat[r, c]
                    # ASSIGN EXACTLY 1 VECTOR PER BROADCAST ROW
                    vecs[idx, id1 * MAX_NODES + id2] = 1.0
                idx += 1

        return counts, vecs, pen

    c_dp, bv_dp, pen_dp = simulate_grid(is_col=flip, apply_pen=not flip)
    c_q, bv_q, pen_q = simulate_grid(is_col=not flip, apply_pen=flip)
    c_sb, c_red, _ = simulate_grid(is_col=not flip, apply_pen=False, is_sb=True)

    node_mask = np.zeros(MAX_NODES, dtype=np.float32)
    for n in node_list:
        node_mask[node_to_id[n]] = 1.0

    if get_pen:
        def get_internode_unit(fid):
            n1 = fid // MAX_NODES
            n2 = fid % MAX_NODES
            return 1.0 if n1 != n2 else 0.0

        dp_sum = 0.0
        q_sum = 0.0

        for fid in range(MAX_NODES * MAX_NODES):
            dp_sum += get_internode_unit(fid) * c_dp[fid]
            q_sum += get_internode_unit(fid) * c_q[fid]

        max_dp_b = 0.0
        max_q_b = 0.0
        for i in range(MAX_BCAST_LINKS):
            dp_b = 0.0
            q_b = 0.0
            for j in range(MAX_NODES * MAX_NODES):
                dp_b += get_internode_unit(j) * bv_dp[i, j]
                q_b += get_internode_unit(j) * bv_q[i, j]
            max_dp_b = max(max_dp_b, dp_b)
            max_q_b = max(max_q_b, q_b)

        dp_sum += max_dp_b
        q_sum += max_q_b
        dp_sum *= local_d
        q_sum *= local_q

        if flip: # query penalty
            pen = (t_q / t_d) / (q_sum / dp_sum)
        else: # data penalty
            pen = (t_d / t_q) / (dp_sum / q_sum)

        return pd.Series({
            'c_dp': c_dp, 'bv_dp': bv_dp, 'pen_dp': pen_dp,
            'c_q': c_q, 'bv_q': bv_q, 'pen_q': pen_q,
            'c_sb': c_sb, 'c_red': c_red, 'node_mask': node_mask,
            'dp_sum': dp_sum, 'q_sum': q_sum, 'actual_penalty' : pen
        })

    return pd.Series({
        'c_dp': c_dp, 'bv_dp': bv_dp, 'pen_dp': pen_dp,
        'c_q': c_q, 'bv_q': bv_q, 'pen_q': pen_q,
        'c_sb': c_sb, 'c_red': c_red, 'node_mask': node_mask
    })

def process_df(df):
    df.rename(columns={'data_row': 'flip'}, inplace=True)
    df['query_split'] = df['p_threads'] // df['data_split']
    df['local_datapoints'] = df['num_datapoints'] / df['data_split']
    df['local_queries'] = df['num_queries'] / df['query_split']

    print("Precomputing topology tensors...")
    topo_df = df.apply(precompute_topology, axis=1)
    df['dp_sum'] = topo_df['dp_sum']
    df['q_sum'] = topo_df['q_sum']
    df['actual_penalty'] = topo_df['actual_penalty']

    return df.drop(labels=['p_threads', 'num_datapoints', 'num_queries'], axis=1), topo_df.drop(labels=['dp_sum', 'q_sum', 'actual_penalty'], axis=1)

def extract_coefficients(df, epochs=100, lr=5e-3):
    print("Training mode-aware analytical model...")
    print("-" * 65)

    df, topo_df = process_df(df)

    feature_cols = [
        'local_datapoints', 'local_queries',
        'num_attrs', 'avg_neighbours',
        'data_split', 'query_split',
        'mode', 'flip', 'nodes'
    ]
    target_cols = ['send_data', 'send_query', 'build_tree', 'computation', 'reduce', 'sendback']

    df_features = df[feature_cols]
    df_targets = df[target_cols]

    # Convert dictionary of Pandas Series to grouped PyTorch Tensors
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def to_tensor(col_name):
        return torch.tensor(np.stack(topo_df[col_name].values), dtype=torch.float32).to(device)

    topo_tensors = {
        'dp': (to_tensor('c_dp'), to_tensor('bv_dp'), to_tensor('pen_dp')),
        'q':  (to_tensor('c_q'), to_tensor('bv_q'), to_tensor('pen_q')),
        'red': (to_tensor('c_red')),
        'sb': (to_tensor('c_sb')),
        'node_mask': to_tensor('node_mask')
    }

    # --- 2. Prepare Numeric Tensors ---
    df_features_numeric = df_features.drop(['nodes'], axis=1)
    df_features_numeric = df_features_numeric.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    df_targets = df_targets.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    X = torch.tensor(df_features_numeric.values, dtype=torch.float32).to(device)
    Y = torch.tensor(df_targets.values, dtype=torch.float32).to(device)

    model = HPCAnalyticalModel().to(device)
    criterion = nn.HuberLoss() # Switched to Huber for robustness against HPC jitter

    # --- OPTIMIZER & SCHEDULER SETUP ---
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Decay the LR by half (0.5) 4 times evenly across the total epochs
    step_size = max(250, epochs // 16)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=0.707)

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Execute fully vectorized forward pass
        predictions = model(X, topo_tensors)

        loss_sd   = criterion(predictions[:, 0], Y[:, 0])
        loss_sq   = criterion(predictions[:, 1], Y[:, 1])
        loss_bt   = criterion(predictions[:, 2], Y[:, 2])
        loss_comp = criterion(predictions[:, 3], Y[:, 3])
        loss_red  = criterion(predictions[:, 4], Y[:, 4])
        loss_sb   = criterion(predictions[:, 5], Y[:, 5])

        def get_norm(idx):
            return torch.clamp(Y[:, idx].detach().mean()**2, min=0.01)

        norm_loss_sd   = loss_sd / get_norm(0)
        norm_loss_sq   = loss_sq / get_norm(1)
        norm_loss_bt   = loss_bt / get_norm(2)
        norm_loss_comp = loss_comp / get_norm(3)
        norm_loss_red   = loss_red / get_norm(4)
        norm_loss_sb   = loss_sb / get_norm(5)

        total_loss = norm_loss_sd + norm_loss_sq + norm_loss_comp + norm_loss_red + norm_loss_sb + norm_loss_bt

        if torch.isnan(total_loss):
            print("\nCRITICAL ERROR: total_loss is NaN before backward pass!")
            break

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        # --- UPDATE LEARNING RATE ---
        scheduler.step()

        with torch.no_grad():
            for param in model.parameters():
                if torch.isnan(param).any():
                    param.masked_fill_(torch.isnan(param), 0.1)
                param.clamp_(min=1e-12)

        if epoch % 100 == 0:
            # Fetch current learning rate for logging
            current_lr = scheduler.get_last_lr()[0]
            print(f"Epoch {epoch:5d} | Loss: {total_loss.item():.4f} | SD: {norm_loss_sd.item():.4f} | SQ: {norm_loss_sq.item():.4f} | RED: {norm_loss_red.item():.4f} | SB: {norm_loss_sb.item():.4f} | LR: {current_lr:.6f}")

    print(f"\nFinal Loss: {total_loss.item():.4f}\n")

    print("\n=== RAW MODEL PARAMETERS ===")
    for name, param in model.named_parameters():
        if param.requires_grad:
            if param.numel() == 1:
                print(f"{name:<15} = {param.data.item():.6f}")
            else:
                print(f"{name:<15} = [Learned Matrix/Array of shape {list(param.shape)}]")

    return model

# --- EXECUTION SCRIPT ---

# Train pairwise latencies

# Intranode bandwidth (only train bandwidths, only backprop dp loss)
# df = pd.concat([pd.read_csv('overview_1.csv'),pd.read_csv('overview_2.csv'),pd.read_csv('comp_spd.csv')])
# df = df[df['nodes'].str.len() == 5]

# # Internode bandwidth (only train bandwidths, only backprop dp loss)
# df = pd.concat([pd.read_csv('bandwidth.csv'), pd.read_csv('benchdata.csv'), pd.read_csv('i7-ultimatum.csv'), pd.read_csv('bench4.csv'),pd.read_csv('bench3.csv')])

# # Coeffs
df = pd.concat([pd.read_csv('bonus.csv')])
# df = df[df['mode'] != 2]

# process_df(df)[0]
# Train the model
# trained_model = extract_coefficients(df, epochs=2000, lr=5e-3)


In [31]:
v = 0
def manual_predict(node_list, N, Q, A, K, d_split, flip, mode, model=HPCAnalyticalModel()):
    """
    Runs a single prediction using the vectorized HPCAnalyticalModel.
    Assumes `precompute_topology` is available in the global scope.
    """
    model.eval() # Set model to evaluation mode
    device = next(model.parameters()).device # Autodetect CPU or CUDA

    # 1. Calculate derived parameters
    total_p_threads = sum(node_to_cores[node_id] for node_id in node_list)
    query_split = total_p_threads // d_split if d_split != 0 else 0.0

    local_datapoints = N / d_split if d_split != 0 else 0.0
    local_queries = Q / query_split if query_split != 0 else 0.0

    # 2. Package data for the topology precomputator
    # We use 'flip' here to ensure compatibility with precompute_topology
    row_data = pd.Series({
        'nodes': "[" + "&".join(node_list) + "]",
        'data_split': float(d_split),
        'query_split': float(query_split),
        'flip': bool(flip)
    })

    # 3. Generate the O(1) Topology Tensors
    topo_series = precompute_topology(row_data, get_pen=False)

    def to_batch_tensor(val):
        # Convert numpy arrays to tensors and add a batch dimension of size 1
        return torch.tensor(val, dtype=torch.float32).unsqueeze(0).to(device)

    topo_tensors = {
        'dp': (to_batch_tensor(topo_series['c_dp']), to_batch_tensor(topo_series['bv_dp']),
               to_batch_tensor(topo_series['pen_dp'])),
        'q':  (to_batch_tensor(topo_series['c_q']), to_batch_tensor(topo_series['bv_q']),
               to_batch_tensor(topo_series['pen_q'])),
        'red': (to_batch_tensor(topo_series['c_red'])),
        'sb': (to_batch_tensor(topo_series['c_sb'])),
        'node_mask': to_batch_tensor(topo_series['node_mask'])
    }

    global v
    v = topo_tensors

    # 4. Construct the numeric feature tensor (Strictly ordered to match forward pass)
    # [N_loc, Q_loc, A, K, S_d, S_q, mode]
    numeric_features = [
        local_datapoints,
        local_queries,
        A,
        K,
        d_split,
        query_split,
        mode
    ]
    x = torch.tensor([numeric_features], dtype=torch.float32).to(device)

    # 5. Execute Prediction
    with torch.no_grad():
        predictions = model(x, topo_tensors)

    return predictions

# --- Test single prediction ---
# Assuming trained_model is your fully trained HPCAnalyticalModel
pred = manual_predict(['031','032'], 7000, 20000, 250, 70, 1, 1, 0)
print(pred, pred.sum())

tensor([1290.1631]) tensor([1814.6195]) tensor([0.]) tensor([1814.6195])
tensor([[0.0661, 0.0929, 0.0000, 1.2596, 0.0000, 1.7453]]) tensor(3.1639)


In [21]:
import torch

def print_tensor_as_cpp(tensor, name="my_tensor", decimals=4):
    """
    Prints a PyTorch tensor as a copy-pasteable C++ std::vector.
    Supports any number of dimensions.
    """
    # Detach and move to CPU to avoid graph/device errors
    val_list = tensor.detach().cpu().tolist()

    # Dynamically build the C++ type based on dimensions
    dims = tensor.dim()
    cpp_type = "float"
    for _ in range(dims):
        cpp_type = f"vector<{cpp_type}>"

    # Recursive formatter
    def build_str(item, depth=1):
        if isinstance(item, list):
            inner = [build_str(sub_item, depth + 1) for sub_item in item]
            indent = "    " * depth

            # If it's a list of lists, use newlines for readability.
            # If it's the bottom-level list of numbers, keep it inline.
            if isinstance(item[0], list):
                return "{\n" + indent + (",\n" + indent).join(inner) + "\n" + "    " * (depth - 1) + "}"
            else:
                return "{" + ", ".join(inner) + "}"
        else:
            # Base case: format number to requested decimals with 'f' suffix
            return f"{item:.{decimals}f}f"

    formatted_str = f"{cpp_type} {name} = {build_str(val_list)};"
    print(formatted_str)
    print("\n" + "="*50 + "\n")


# node_comp_bf = trained_model.node_comp_bf.data
# node_comp_kd = trained_model.node_comp_kd.data
# node_pair_bandwidth = trained_model.node_pair_bandwidth.data

print_tensor_as_cpp(node_comp_bf, name="node_comp_bf")
print_tensor_as_cpp(node_comp_kd, name="node_comp_kd")
print_tensor_as_cpp(node_pair_bandwidth, name="node_pair_bandwidth")

vector<float> node_comp_bf = {1.6207f, 1.1694f, 1.1694f, 1.1694f, 1.5697f, 1.5697f, 0.6403f, 1.5281f, 1.5281f, 1.5281f, 1.5281f, 1.5281f, 0.9086f, 0.9086f, 0.9086f, 0.9086f, 0.9086f, 0.9086f, 0.9086f, 0.9086f};


vector<float> node_comp_kd = {0.5386f, 0.9750f, 0.9750f, 0.9750f, 0.8193f, 0.8193f, 0.3643f, 1.9103f, 1.9103f, 1.9103f, 1.9103f, 1.9103f, 0.5825f, 0.5825f, 0.5825f, 0.5825f, 0.5825f, 0.5825f, 0.5825f, 0.5825f};


vector<vector<float>> node_pair_bandwidth = {
    {0.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f},
    {1.0000f, 0.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f},
    {1.0000f, 1.0000f, 0.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f, 1.0000f

# Visualizer

In [84]:
import matplotlib.pyplot as plt
import numpy as np

def analyze_nodes(node_list, d_split, N_size, Q_size, flip=False):
    # ── 1. Configuration ──────────────────────────────────────────────────────
    pcore_map = {
        "xs-4114": 20, "i7-7700": 8, "i7-9700": 8,
        "i7-13700": 16, "w5-3423": 24, "dxs-4114": 40
    }
    partition_nodes = {
        "xs-4114":  "008",
        "i7-7700":  "015 016 012",
        "dxs-4114": "018 019",
        "i7-9700":  "020",
        "w5-3423":  "029 030 031 032 027",
        "i7-13700": "034 035 033 036 037 038 039 040"
    }

    # ── ANSI helpers ──────────────────────────────────────────────────────────
    RESET   = "\033[0m";  BOLD    = "\033[1m";  DIM     = "\033[2m"
    RED     = "\033[91m"; GREEN   = "\033[92m"; YELLOW  = "\033[93m"
    CYAN    = "\033[96m"; BLUE    = "\033[94m"; MAGENTA = "\033[95m"
    WHITE   = "\033[97m"; ORANGE  = "\033[38;5;209m"

    NODE_COLORS = ["\033[42m", "\033[44m", "\033[45m", "\033[43m",
                   "\033[41m", "\033[46m"]

    def box(title, width=68):
        bar = "─" * (width - 2)
        pad = (width - 2 - len(title)) // 2
        print(f"\n{CYAN}┌{bar}┐{RESET}")
        print(f"{CYAN}│{RESET}{' '*pad}{BOLD}{WHITE}{title}{RESET}"
              f"{' '*(width-2-pad-len(title))}{CYAN}│{RESET}")
        print(f"{CYAN}└{bar}┘{RESET}")

    def section(title, color=BLUE):
        print(f"\n{color}{BOLD}  ▶  {title}{RESET}")
        print(f"{DIM}  {'─'*60}{RESET}")

    def kv(label, value, unit="", color=WHITE):
        print(f"  {DIM}{label:<34}{RESET} {color}{BOLD}{value}{RESET} {DIM}{unit}{RESET}")

    def bar_chart(label, value, max_val, width=28, color=CYAN):
        filled = int((value / max_val) * width) if max_val > 0 else 0
        bar    = "█" * filled + "░" * (width - filled)
        pct    = (value / max_val * 100) if max_val > 0 else 0
        print(f"  {label:<22} {color}{bar}{RESET}  {BOLD}{value:.5f}s{RESET}  "
              f"{DIM}({pct:.1f}%){RESET}")

    def dual_timeline(ta_scatter, ta_bcast, tb_scatter, tb_bcast, label_a, label_b):
        """Draws two parallel timelines aligned to the same total width."""
        ta = ta_scatter + ta_bcast
        tb = tb_scatter + tb_bcast
        t_max = max(ta + tb, 1e-12)
        W = 48
        def render_lane(ts, tb_val, total, name, sc, bc, end_marker=""):
            sw = int((ts / t_max) * W)
            bw = int((tb_val / t_max) * W)
            rest = W - sw - bw
            s_bar = f"{sc}{'━'*sw}{RESET}"
            b_bar = f"{bc}{'━'*bw}{RESET}"
            r_bar = f"{DIM}{'╌'*rest}{RESET}" if rest > 0 else ""
            flag  = f" {RED}{BOLD}<< critical{RESET}" if end_marker else ""
            print(f"  {name:<12} {s_bar}{b_bar}{r_bar} {DIM}{total:.5f}s{RESET}{flag}")
        print()
        render_lane(ta_scatter, ta_bcast, ta, label_a, GREEN,  MAGENTA,
                    end_marker=(ta >= tb))
        render_lane(tb_scatter, tb_bcast, tb, label_b, ORANGE,  CYAN,
                    end_marker=(tb >  ta))
        print(f"  {DIM}             {'─'*W}{RESET}")
        print(f"  {DIM}             {GREEN}████{RESET}{DIM}DB-sctr  "
              f"{MAGENTA}████{RESET}{DIM}=DB-bcast  "
              f"{ORANGE}████{RESET}{DIM}=Q-sctr  "
              f"{CYAN}████{RESET}{DIM}=Q-bcast{RESET}")

    # ── 2. Build matrix ───────────────────────────────────────────────────────
    incontiguous_penalty = 1
    master_node = node_list[0]

    node_to_hw = {
        node: hw for hw, nodes in partition_nodes.items()
        for node in nodes.split()
    }

    flat_ids = []
    for node_id in node_list:
        cores = pcore_map.get(node_to_hw.get(node_id, ""), 0)
        flat_ids.extend([node_list.index(node_id)] * cores)
        if node_id != master_node:
            incontiguous_penalty = max(incontiguous_penalty, cores)

    total_pcores = len(flat_ids)
    if (d_split == 1 and not flip) or (flip and d_split == total_pcores):
        incontiguous_penalty = 1

    rows, cols   = (d_split, total_pcores // d_split) if flip \
                   else (total_pcores // d_split, d_split)
    matrix = np.array(flat_ids).reshape(rows, cols)

    # ── 3. Communication helpers ──────────────────────────────────────────────
    S_SCALE = 1.4e-5
    B_SCALE = 1.4e-5

    def get_lat(src_idx, dst_idx):
        src, dst = node_list[int(src_idx)], node_list[int(dst_idx)]
        return pairings[(src, dst)]

    # ── 4. Overview ───────────────────────────────────────────────────────────
    orient = "flip=Rows (d_split defines row count)" \
             if flip else "flip=Cols (d_split defines col count)"
    box(f"  analyze_nodes  |  {orient}  ")

    section("Input Configuration", CYAN)
    kv("Node list",              str(node_list),             color=CYAN)
    kv("d_split",                d_split,                    color=YELLOW)
    kv("N_size  (database rows)", f"{N_size:,}",             color=GREEN)
    kv("Q_size  (query rows)",    f"{Q_size:,}",             color=ORANGE)
    kv("Total p-cores",           total_pcores,              color=WHITE)
    kv("Matrix shape",            f"{rows} rows x {cols} cols", color=WHITE)

    # Pipeline axis description
    if not flip:
        kv("DB   pipeline", "scatter ROW-0 (horiz) -> broadcast each COL (vert)", color=GREEN)
        kv("Query pipeline", "scatter COL-0 (vert)  -> broadcast each ROW (horiz)", color=ORANGE)
    else:
        kv("DB   pipeline", "scatter COL-0 (vert)  -> broadcast each ROW (horiz)", color=GREEN)
        kv("Query pipeline", "scatter ROW-0 (horiz) -> broadcast each COL (vert)", color=ORANGE)

    section("Node Hardware Details", BLUE)
    print(f"  {'Node ID':<10} {'Hardware':<14} {'P-Cores':<10} {'Idx'}")
    print(f"  {DIM}{'─'*9} {'─'*13} {'─'*9} {'─'*6}{RESET}")
    for i, nid in enumerate(node_list):
        hw    = node_to_hw.get(nid, "unknown")
        cores = pcore_map.get(hw, 0)
        color = NODE_COLORS[i % len(NODE_COLORS)]
        print(f"  {color} {nid} {RESET}  {hw:<14} {cores:<10} idx={i}")

    # ── 5. ASCII Matrix ───────────────────────────────────────────────────────
    section("Core Assignment Matrix", BLUE)

    for r in range(rows):
        # Row label — mark DB scatter root row (not flip) or Q scatter root row (flip)
        if not flip and r == 0:
            row_lbl = f"{GREEN}{BOLD}R{r}* {RESET}"
        elif flip and r == 0:
            row_lbl = f"{ORANGE}{BOLD}R{r}* {RESET}"
        else:
            row_lbl = f"{DIM}R{r}  {RESET}"

        cells = []
        for c in range(cols):
            nidx  = int(matrix[r, c])
            color = NODE_COLORS[nidx % len(NODE_COLORS)]
            label = node_list[nidx]
            # Mark DB scatter root col (flip) or Q scatter root col (not flip)
            if flip and c == 0:
                mk = f"{GREEN}*{RESET}"
            elif not flip and c == 0:
                mk = f"{ORANGE}*{RESET}"
            else:
                mk = " "
            cells.append(f"{color}{BOLD}{label}{RESET}{mk}")

        print(f"  {row_lbl}" + " ".join(cells))

    print(f"\n  {DIM}Legend:{RESET}")
    for i, nid in enumerate(node_list):
        hw    = node_to_hw.get(nid, "?")
        color = NODE_COLORS[i % len(NODE_COLORS)]
        print(f"    {color} {nid} {RESET}  node {i}  ({hw})")
    print(f"  {GREEN}*{RESET} = DB scatter root axis   "
          f"{ORANGE}*{RESET} = Query scatter root axis")

    # ── Helper: run one scatter → broadcast pipeline ──────────────────────────
    def run_pipeline(name, size, scatter_axis, color_s, color_b):
        """
        scatter_axis = 'row'  -> scatter across Row 0, broadcast down each col
        scatter_axis = 'col'  -> scatter down Col 0, broadcast across each row
        size = N_size or Q_size exclusively
        Returns (t_scatter, t_bcast, lane_times, slowest_lane)
        """
        print(f"\n  {color_s}{BOLD}[{name}]  size={size:,}{RESET}")

        # ---- Phase 1: Scatter ------------------------------------------------
        s_lats = []
        if scatter_axis == "row":
            severity = size / cols
            print(f"  {DIM}  Scatter Row-0 -> all cols  |  "
                  f"severity = {size}/{cols} = {severity:,.1f}{RESET}")
            root_id = matrix[0, 0]
            for c in range(cols):               # no skip — includes c==0 (self)
                dest_id = matrix[0, c]
                lat  = get_lat(root_id, dest_id)
                t    = severity * lat * S_SCALE
                s_lats.append(t)
                src_n = node_list[int(root_id)]
                dst_n = node_list[int(dest_id)]
                tag   = f"{DIM}(intranode){RESET}" if lat == 0 else ""
                print(f"    Scatter C{c:<3} [{src_n}]--({lat:.1f}ms)--> "
                      f"[{dst_n}]  t={t:.5f}s {tag}")
        else:  # col
            severity = size / rows
            print(f"  {DIM}  Scatter Col-0 -> all rows  |  "
                  f"severity = {size}/{rows} = {severity:,.1f}{RESET}")
            root_id = matrix[0, 0]
            for r in range(rows):               # no skip — includes r==0 (self)
                dest_id = matrix[r, 0]
                lat  = get_lat(root_id, dest_id)
                t    = severity * lat * S_SCALE
                s_lats.append(t)
                src_n = node_list[int(root_id)]
                dst_n = node_list[int(dest_id)]
                tag   = f"{DIM}(intranode){RESET}" if lat == 0 else ""
                print(f"    Scatter R{r:<3} [{src_n}]--({lat:.1f}ms)--> "
                      f"[{dst_n}]  t={t:.5f}s {tag}")

        t_scatter = sum(s_lats) if s_lats else 0

        # ---- Phase 2: Broadcast ----------------------------------------------
        lane_times   = []
        lane_details = []

        if scatter_axis == "row":
            # After row-scatter each col head has data -> broadcast down the col
            bcast_severity = size / cols
            print(f"\n  {DIM}  Broadcast down each col (parallel)  |  "
                  f"severity = {size}/{cols} = {bcast_severity:,.1f}{RESET}")
            for c in range(cols):
                lane_root = matrix[0, c]
                lane_max  = 0
                info      = []
                for r in range(rows):           # no skip — includes r==0 (self)
                    dest = matrix[r, c]
                    lat  = get_lat(lane_root, dest)
                    t    = bcast_severity * lat * B_SCALE * incontiguous_penalty
                    lane_max = max(lane_max, t)
                    src_n = node_list[int(lane_root)]
                    dst_n = node_list[int(dest)]
                    tag   = "(intranode)" if lat == 0 else ""
                    info.append(
                        f"      R{r}: [{src_n}]--({lat:.1f}ms)-->[{dst_n}]"
                        f"  t={t:.5f}s {tag}"
                    )
                lane_times.append(lane_max)
                lane_details.append((f"Col {c}", info))
        else:
            # After col-scatter each row head has data -> broadcast across the row
            bcast_severity = size / rows
            print(f"\n  {DIM}  Broadcast across each row (parallel)  |  "
                  f"severity = {size}/{rows} = {bcast_severity:,.1f}{RESET}")
            for r in range(rows):
                lane_root = matrix[r, 0]
                lane_max  = 0
                info      = []
                for c in range(cols):           # no skip — includes c==0 (self)
                    dest = matrix[r, c]
                    lat  = get_lat(lane_root, dest)
                    t    = bcast_severity * lat * B_SCALE
                    lane_max = max(lane_max, t)
                    src_n = node_list[int(lane_root)]
                    dst_n = node_list[int(dest)]
                    tag   = "(intranode)" if lat == 0 else ""
                    info.append(
                        f"      C{c}: [{src_n}]--({lat:.1f}ms)-->[{dst_n}]"
                        f"  t={t:.5f}s {tag}"
                    )
                lane_times.append(lane_max)
                lane_details.append((f"Row {r}", info))

        t_bcast        = max(lane_times) if lane_times else 0
        slowest_idx    = int(np.argmax(lane_times)) if lane_times else 0

        for i, (lname, info) in enumerate(lane_details):
            is_bn  = (i == slowest_idx and t_bcast > 0)
            marker = f" {RED}{BOLD}<-- BOTTLENECK{RESET}" if is_bn else ""
            lcolor = RED if is_bn else color_b
            print(f"  {lcolor}{BOLD}  {lname}{RESET}  "
                  f"max={lane_times[i]:.5f}s{marker}")
            for line in info:
                print(f"{DIM}{line}{RESET}")

        return t_scatter, t_bcast, lane_times, slowest_idx

    # ── 6. Run Both Pipelines ─────────────────────────────────────────────────
    section("DATABASE Pipeline  (N_size only)", GREEN)
    dp_scatter_axis = "row" if not flip else "col"
    t_dp_s, t_dp_b, dp_lane_times, dp_bn = run_pipeline(
        "DATABASE", N_size, dp_scatter_axis, GREEN, MAGENTA
    )

    section("QUERY Pipeline  (Q_size only)", ORANGE)
    q_scatter_axis = "col" if not flip else "row"
    t_q_s, t_q_b, q_lane_times, q_bn = run_pipeline(
        "QUERY", Q_size, q_scatter_axis, ORANGE, CYAN
    )

    # ── 7. Critical Path Summary ──────────────────────────────────────────────
    t_dp    = t_dp_s + t_dp_b
    t_q     = t_q_s  + t_q_b
    t_total = t_dp + t_q
    t_dp_for_bar = max(t_dp, 1e-12)
    t_q_for_bar = max(t_q, 1e-12)

    section("Critical Path Summary", CYAN)

    dp_lane_lbl = ("Col" if not flip else "Row")
    q_lane_lbl  = ("Row" if not flip else "Col")

    print(f"\n  {GREEN}{BOLD}Database pipeline:{RESET}")
    bar_chart("  Scatter (DB)",   t_dp_s, t_dp_for_bar, color=GREEN)
    bar_chart("  Broadcast (DB)", t_dp_b, t_dp_for_bar, color=MAGENTA)
    bar_chart("  DB Total",       t_dp,   t_dp_for_bar, color=YELLOW)

    print(f"\n  {ORANGE}{BOLD}Query pipeline:{RESET}")
    bar_chart("  Scatter (Q)",    t_q_s,  t_q_for_bar, color=ORANGE)
    bar_chart("  Broadcast (Q)",  t_q_b,  t_q_for_bar, color=CYAN)
    bar_chart("  Q Total",        t_q,    t_q_for_bar, color=YELLOW)

    dual_timeline(t_dp_s, t_dp_b, t_q_s, t_q_b, "DB   ", "Query")

    print()
    kv("DB    scatter",   f"{t_dp_s:.5f}",  "s", GREEN)
    kv("DB    broadcast", f"{t_dp_b:.5f}",  "s", MAGENTA)
    kv("DB    total",     f"{t_dp:.5f}",    "s", YELLOW)
    kv("Query scatter",   f"{t_q_s:.5f}",   "s", ORANGE)
    kv("Query broadcast", f"{t_q_b:.5f}",   "s", CYAN)
    kv("Query total",     f"{t_q:.5f}",     "s", YELLOW)
    kv("Combined total",  f"{t_total:.5f}", "s", BLUE)

    t_color = GREEN if t_total < 0.01 else (YELLOW if t_total < 0.1 else RED)
    critical = f"{GREEN}DB{RESET}" if t_dp >= t_q else f"{ORANGE}Query{RESET}"
    print(f"\n  {DIM}{'─'*60}{RESET}")
    kv("CRITICAL PIPELINE",      "", "", color=t_color)
    print(f"    -> {critical} {BOLD}{max(t_dp, t_q):.5f}s{RESET} "
          f"{DIM}(other={min(t_dp,t_q):.5f}s){RESET}")
    kv("DB    bottleneck lane",
       f"{dp_lane_lbl} {dp_bn}", "", RED)
    kv("Query bottleneck lane",
       f"{q_lane_lbl} {q_bn}",  "", RED)
    print(f"\n{CYAN}{'─'*68}{RESET}\n")

    return {
        "t_dp_scatter":   t_dp_s, "t_dp_bcast":    t_dp_b, "t_dp_total":   t_dp,
        "t_q_scatter":    t_q_s,  "t_q_bcast":     t_q_b,  "t_q_total":    t_q,
        "t_total":        t_total,
        "dp_bottleneck_lane": dp_bn, "q_bottleneck_lane": q_bn,
        "dp_lane_times":  dp_lane_times, "q_lane_times": q_lane_times,
    }


# ── Example Usage ─────────────────────────────────────────────────────────────

analyze_nodes(["031", "032"], d_split=4, N_size=90000, Q_size=10000, flip=True)



┌──────────────────────────────────────────────────────────────────┐
│     analyze_nodes  |  flip=Rows (d_split defines row count)      │
└──────────────────────────────────────────────────────────────────┘

  ▶  Input Configuration
  ────────────────────────────────────────────────────────────
  Node list                          ['031', '032'] 
  d_split                            4 
  N_size  (database rows)            90,000 
  Q_size  (query rows)               10,000 
  Total p-cores                      48 
  Matrix shape                       4 rows x 12 cols 
  DB   pipeline                      scatter COL-0 (vert)  -> broadcast each ROW (horiz) 
  Query pipeline                     scatter ROW-0 (horiz) -> broadcast each COL (vert) 

  ▶  Node Hardware Details
  ────────────────────────────────────────────────────────────
  Node ID    Hardware       P-Cores    Idx
  ───────── ───────────── ───────── ──────
   031   w5-3423        24         idx=0
   032   w5-3423        24 

NameError: name 'pairings' is not defined

In [ ]:
import numpy as np

def analyze_nodes(SCALE, node_list, d_split, N_size, Q_size, flip=False):
    # ── 1. Configuration ──────────────────────────────────────────────────────
    pcore_map = {
        "xs-4114": 20, "i7-7700": 8, "i7-9700": 8,
        "i7-13700": 16, "w5-3423": 24, "dxs-4114": 40
    }
    partition_nodes = {
        "xs-4114":  "008",
        "i7-7700":  "015 016 012",
        "dxs-4114": "018 019",
        "i7-9700":  "020",
        "w5-3423":  "029 030 031 032 027",
        "i7-13700": "034 035 033 036 037 038 039 040"
    }

    master_node = node_list[0]
    incontiguous_penalty = 1

    node_to_hw = {
        node: hw for hw, nodes in partition_nodes.items()
        for node in nodes.split()
    }

    flat_ids = []
    for node_id in node_list:
        cores = pcore_map.get(node_to_hw.get(node_id, ""), 0)
        flat_ids.extend([node_list.index(node_id)] * cores)
        if node_id != master_node:
            incontiguous_penalty = max(incontiguous_penalty, cores)

    total_pcores = len(flat_ids)
    if (not flip and d_split == 1) or (flip and d_split == total_pcores):
        incontiguous_penalty = 1

    rows, cols   = (d_split, total_pcores // d_split) if flip \
                   else (total_pcores // d_split, d_split)
    matrix = np.array(flat_ids).reshape(rows, cols)

    # ── 3. Communication helpers ──────────────────────────────────────────────

    def get_lat(src_idx, dst_idx):
        src, dst = node_list[int(src_idx)], node_list[int(dst_idx)]
        return pairings[(src, dst)]


    # ── Helper: run one scatter → broadcast pipeline ──────────────────────────
    def run_pipeline(name, size, scatter_axis):
        """
        scatter_axis = 'row'  -> scatter across Row 0, broadcast down each col
        scatter_axis = 'col'  -> scatter down Col 0, broadcast across each row
        size = N_size or Q_size exclusively
        Returns (t_scatter, t_bcast, lane_times, slowest_lane)
        """

        # ---- Phase 1: Scatter ------------------------------------------------
        s_lats = []
        if scatter_axis == "row":
            severity = size / cols
            root_id = matrix[0, 0]
            for c in range(1, cols):               # no skip — includes c==0 (self)
                dest_id = matrix[0, c]
                lat  = get_lat(root_id, dest_id)
                t    = severity * lat * SCALE
                s_lats.append(t)
        else:  # col
            severity = size / rows
            root_id = matrix[0, 0]
            for r in range(1, rows):               # no skip — includes r==0 (self)
                dest_id = matrix[r, 0]
                lat  = get_lat(root_id, dest_id)
                t    = severity * lat * SCALE
                s_lats.append(t)

        t_scatter = sum(s_lats) if s_lats else 0

        # ---- Phase 2: Broadcast ----------------------------------------------
        lane_times   = []

        if scatter_axis == "row":
            # After row-scatter each col head has data -> broadcast down the col
            bcast_severity = size / cols
            for c in range(cols):
                lane_root = matrix[0, c]
                lane_max  = 0
                for r in range(rows):           # no skip — includes r==0 (self)
                    dest = matrix[r, c]
                    lat  = get_lat(lane_root, dest)
                    t    = bcast_severity * lat * SCALE * incontiguous_penalty
                    lane_max = max(lane_max, t)
                lane_times.append(lane_max)
        else:
            # After col-scatter each row head has data -> broadcast across the row
            bcast_severity = size / rows
            for r in range(rows):
                lane_root = matrix[r, 0]
                lane_max  = 0
                for c in range(cols):           # no skip — includes c==0 (self)
                    dest = matrix[r, c]
                    lat  = get_lat(lane_root, dest)
                    t    = bcast_severity * lat * SCALE
                    lane_max = max(lane_max, t)
                lane_times.append(lane_max)

        # Result is the maximum time taken by any single broadcast lane
        total_bcast_estimate = max(lane_times) if lane_times else 0

        t_bcast        = max(lane_times) if lane_times else 0
        slowest_idx    = int(np.argmax(lane_times)) if lane_times else 0

        return t_scatter, t_bcast, lane_times, slowest_idx

    # ── 6. Run Both Pipelines ─────────────────────────────────────────────────
    dp_scatter_axis = "row" if not flip else "col"
    t_dp_s, t_dp_b, dp_lane_times, dp_bn = run_pipeline(
        "DATABASE", N_size, dp_scatter_axis
    )
    q_scatter_axis = "col" if not flip else "row"
    t_q_s, t_q_b, q_lane_times, q_bn = run_pipeline(
        "QUERY", Q_size, q_scatter_axis
    )

    # ── 7. Critical Path Summary ──────────────────────────────────────────────
    t_dp    = t_dp_s + t_dp_b
    t_q     = t_q_s  + t_q_b
    t_total = t_dp + t_q          # parallel pipelines

    return {
        "t_dp_scatter":   t_dp_s, "t_dp_bcast":    t_dp_b, "t_dp_total":   t_dp,
        "t_q_scatter":    t_q_s,  "t_q_bcast":     t_q_b,  "t_q_total":    t_q,
        "t_total":        t_total,
        "dp_bottleneck_lane": dp_bn, "q_bottleneck_lane": q_bn,
        "dp_lane_times":  dp_lane_times, "q_lane_times": q_lane_times,
    }


# ── Example Usage ─────────────────────────────────────────────────────────────
analyze_nodes(1.4e-5, ["034", "035"], d_split=4, N_size=50000, Q_size=50000)

{'t_dp_scatter': 0.00111686571315711,
 't_dp_bcast': 2.800000070912109,
 't_dp_total': 2.801116936625266,
 't_q_scatter': 0.35055844172059214,
 't_q_bcast': 0.000186144285526185,
 't_q_total': 0.3507445860061183,
 't_total': 3.1518615226313846,
 'dp_bottleneck_lane': 0,
 'q_bottleneck_lane': 0,
 'dp_lane_times': [2.800000070912109,
  2.800000070912109,
  2.800000070912109,
  2.800000070912109],
 'q_lane_times': [0.000186144285526185,
  0.000186144285526185,
  0.000186144285526185,
  0.000186144285526185,
  0.0001595522447367275,
  0.0001595522447367275,
  0.0001595522447367275,
  0.0001595522447367275]}

In [ ]:
df= pd.read_csv('comp_spd.csv')

In [ ]:

print(df[(df['num_attrs'] == 10) & (df['mode'] == 0)]['computation'].mean())
print(df[(df['num_attrs'] == 10) & (df['mode'] == 1)]['computation'].mean())

0.6275166666666667
0.6147499999999998


# 3D Plot

In [ ]:
import plotly.express as px
import pandas as pd

df = pd.concat([pd.read_csv('optimal_split_4.csv'),pd.read_csv('optimal_split_4b.csv')])
df = df[(df['data_row'] == 0)]

In [ ]:
fig = px.scatter_3d(df, x='data_split', y='num_datapoints', z='send_query')
fig.show()

# Predictor

In [32]:
import numpy as np

def find_best_topology(node_list, N_size, Q_size, A, K):
    """
    Evaluates all matrix permutations for given nodes to find the optimal split
    and orientation that minimizes total communication time.
    """

    # ── 1. One-Time Hardware Mapping ──────────────────────────────────────────
    flat_ids = []
    max_remote_cores = 1
    for i, node in enumerate(node_list):
        cores = node_to_cores.get(node, 16) # Default to 16 if unknown
        flat_ids.extend([i] * cores)
        if i > 0:
            max_remote_cores = max(max_remote_cores, cores)

    total_pcores = len(flat_ids)

    # ── 2. Internal Helpers ───────────────────────────────────────────────────
    def get_factors(num):
        """Finds all integer factors of num."""
        factors, back = [], []
        for i in range(1, int(num**0.5) + 1):
            if num % i == 0:
                factors.append(i)
                if i != num // i:  # Prevents duplicating the square root
                    back.append(num // i)
        factors.extend(back[::-1])
        return factors


    print(f"Total Cores: {total_pcores} | Possible Splits: {get_factors(total_pcores)}")
    # print("-" * 65)

    # Search space: all factors of total_pcores
    for split in get_factors(total_pcores):
        for flip_state in [True, False]:
            print("config split", split, "flip", flip_state)
            times = manual_predict(node_list, N_size, Q_size, A, K, split, flip_state, 0)
            t_d, t_q, _, _, t_red, t_sb = times[0]
            print("t_d:", t_d, "t_q:", t_q, "t_red:", t_red, "t_sb:", t_sb,'\n')


print(find_best_topology(['031', '032'], 7000, 20000, 375, 70))

Total Cores: 48 | Possible Splits: [1, 2, 3, 4, 6, 8, 12, 16, 24, 48]
config split 1 flip True
tensor([1290.1631]) tensor([1814.6195]) tensor([0.]) tensor([1814.6195])
t_d: tensor(0.0990) t_q: tensor(0.1393) t_red: tensor(0.) t_sb: tensor(1.7453) 

config split 1 flip False
tensor([1290.1631]) tensor([1814.6195]) tensor([0.]) tensor([1814.6195])
t_d: tensor(0.0990) t_q: tensor(0.1393) t_red: tensor(0.) t_sb: tensor(1.7453) 

config split 2 flip True
tensor([645.0815]) tensor([3629.2393]) tensor([96.6500]) tensor([1309.6390])
t_d: tensor(0.0495) t_q: tensor(0.2786) t_red: tensor(0.0056) t_sb: tensor(1.2596) 

config split 2 flip False
tensor([15721.1074]) tensor([1843.0898]) tensor([56.9408]) tensor([1786.1490])
t_d: tensor(1.2066) t_q: tensor(0.1415) t_red: tensor(0.0034) t_sb: tensor(1.7179) 

config split 3 flip True
tensor([860.1086]) tensor([6810.4385]) tensor([289.9500]) tensor([1281.1687])
t_d: tensor(0.0660) t_q: tensor(0.5227) t_red: tensor(0.0147) t_sb: tensor(1.2322) 

config